# 한국 대장주 로테이션 백테스트

**시총 상위 100위 이내에서 12개월 모멘텀 상위 10종목을 매월 동일비중으로 보유하고, KOSPI가 200일 이동평균 아래로 내려가면 전량 현금으로 물러나는 전략**을 2000년 이후 KRX 전종목 데이터로 검증합니다.

| | CAGR | MDD | 샤프 |
|---|---|---|---|
| 이 전략 | 17.9% | −51% | 0.77 |
| KOSPI 매수후보유 | 7.2% | −56% | 0.41 |

### 어떻게 여기까지 왔나

이 노트북은 유튜브 쇼츠의 트레이딩 전략(트레이더 오스틴 실버 — *"고점 대비 −20% 빠질 때까지 기다렸다가, 저점에서 하락폭의 절반을 회복하면 매수"*)을 한국 대장주에 백테스트한 데서 출발했습니다. 결과는:

- 그 **진입 타이밍에는 정보가 없었습니다.** 같은 종목·같은 거래횟수·같은 보유일수를 무작위 시점에 배치한 몬테카를로 분포에서 **백분위 55** — 사실상 한가운데였습니다.
- 성과를 가른 것은 **무엇을 들고 있었나**였습니다. 2000년 대장주 7개 중 삼성전자는 +4,033%, 현대차는 +1,654%였지만 당시 시총 1위 KT는 −68%, 4위 SK텔레콤은 −77%였습니다.

그래서 타이밍 대신 **종목선정**을 규칙으로 만든 것이 이 전략입니다. 부록에서 원래 영상 전략과 직접 비교할 수 있습니다.

### 먼저 읽어주세요

- **알파의 상당 부분이 2000~2013년에서 나왔습니다.** 2013년 7월 이후로는 **KOSPI에 집니다** — 전략 CAGR 9.3% / 샤프 0.49 vs KOSPI 10.2% / 0.57.
- MDD −51%는 실제로 겪기 힘든 수준입니다. 전액이 아니라 **주식자산의 ~70%만** 배분하면 MDD를 −40% 수준으로 낮추면서 수익은 유지됩니다.
- 연 회전율 약 380%. 일반 계좌에서는 세금이 성과를 더 깎습니다.
- 배당 미반영(전략·벤치마크 모두), 슬리피지·유동성 제약 미반영.

**실행 방법**: 상단 메뉴 `런타임 → 모두 실행` (Ctrl+F9). 데이터 다운로드 포함 5~10분 걸립니다.

> 투자 판단의 근거가 아니라 방법론 검증용입니다.

## 1. 환경 준비

In [ ]:
#@title 패키지 설치 · 한글 폰트 { display-mode: "form" }
!pip install -q FinanceDataReader pyarrow 2>/dev/null
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import warnings, matplotlib, matplotlib.pyplot as plt
import matplotlib.font_manager as fm
warnings.filterwarnings("ignore")

try:                                   # 폰트 캐시 재구축 (런타임 재시작 불필요)
    fm._load_fontmanager(try_read_cache=False)
except Exception:
    fm.fontManager.__init__()

for cand in ("NanumGothic", "NanumBarunGothic", "DejaVu Sans"):
    if any(f.name == cand for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = cand
        break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트:", plt.rcParams["font.family"][0])

## 2. 데이터 다운로드

[FinanceData/marcap](https://github.com/FinanceData/marcap) — KRX 전종목 일별 시가총액·주가.
**상장폐지 종목이 포함**되어 있어 생존편향 없이 백테스트할 수 있습니다. 약 490MB.

In [ ]:
#@title KRX 전종목 데이터 (2000~2026)
import os, time, urllib.request, concurrent.futures as cf

CACHE = "marcap"
RAW = "https://raw.githubusercontent.com/FinanceData/marcap/master/data/marcap-{}.parquet"
YEARS = list(range(2000, 2027))
os.makedirs(CACHE, exist_ok=True)

def fetch(year):
    path = f"{CACHE}/marcap-{year}.parquet"
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        return year, "캐시"
    for attempt in range(3):
        try:
            urllib.request.urlretrieve(RAW.format(year), path)
            return year, "다운로드"
        except Exception as exc:
            if attempt == 2:
                return year, f"실패 {exc}"
            time.sleep(2)

t0 = time.time()
with cf.ThreadPoolExecutor(8) as ex:
    results = list(ex.map(fetch, YEARS))
bad = [r for r in results if r[1].startswith("실패")]
size = sum(os.path.getsize(f"{CACHE}/{f}") for f in os.listdir(CACHE)) / 1e6
print(f"{len(YEARS) - len(bad)}/{len(YEARS)}개 연도 · {size:.0f}MB · {time.time() - t0:.0f}초")
if bad:
    print("실패:", bad)

## 3. 데이터 로딩과 수정주가

`Close`는 액면분할이 반영되지 않은 원주가입니다. 주식수가 크게 변했는데 시가총액은 그대로인 날을
분할·병합으로 판정해 보정합니다 (삼성전자 50:1, 카카오 5:1, 하이닉스 21:1 감자 등).
인적분할(NAVER→NHN엔터, 삼성바이오→에피스홀딩스)은 존속법인 기준으로만 반영되어
**실제 주주 수익률보다 보수적**입니다.

In [ ]:
#@title 로딩 · 수정주가 · 행렬 구축
import glob, re
import numpy as np
import pandas as pd

EXCLUDE = re.compile(r"우[BC]?\)?$|우선|스팩|SPAC|리츠$")   # 우선주·스팩·리츠 제외
MATRIX_RANK = 300      # 시총 300위 안에 한 번이라도 든 종목만 행렬로
MIN_HISTORY = 250      # 상장 후 최소 거래일

cols = ["Code", "Name", "Close", "Marcap", "Stocks", "Market", "Date", "Amount"]
frames = []
for f in sorted(glob.glob(f"{CACHE}/marcap-*.parquet")):
    d = pd.read_parquet(f, columns=cols)
    d = d[d["Market"].isin(["KOSPI", "KOSDAQ", "KOSDAQ GLOBAL"])]
    frames.append(d[~d["Name"].str.contains(EXCLUDE, na=False, regex=True)])
df = pd.concat(frames, ignore_index=True)
del frames
df["Code"] = df["Code"].astype(str).str.zfill(6)
print(f"{len(df):,}행 · {df['Code'].nunique():,}종목 · "
      f"{df['Date'].min().date()} ~ {df['Date'].max().date()}")

def build_matrices(df):
    rank = df.groupby("Date")["Marcap"].rank(ascending=False)
    keep = set(df.loc[rank <= MATRIX_RANK, "Code"].unique())
    d = df[df["Code"].isin(keep)].sort_values(["Code", "Date"]).copy()

    grp = d.groupby("Code", sort=False)
    s = (d["Stocks"] / grp["Stocks"].shift(1)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    c = (d["Close"] / grp["Close"].shift(1)).replace([np.inf, -np.inf], np.nan).fillna(1.0)
    split = ((s * c - 1.0).abs() < 0.15) & ((s > 1.4) | (s < 0.72))
    d["ret"] = c * np.where(split, s, 1.0) - 1.0
    d.loc[grp.cumcount() == 0, "ret"] = 0.0
    d["adj"] = grp["ret"].transform(lambda x: (1 + x).cumprod())

    return (d.pivot_table(index="Date", columns="Code", values="adj"),
            d.pivot_table(index="Date", columns="Code", values="Marcap"),
            d.pivot_table(index="Date", columns="Code", values="Amount"),
            d.groupby("Code")["Name"].last())

px, mc, am, names = build_matrices(df)
print(f"행렬: {px.shape[1]:,}종목 × {px.shape[0]:,}거래일")

In [ ]:
#@title 벤치마크(KOSPI)와 현금 금리
import FinanceDataReader as fdr

_k = fdr.DataReader("KS11", "1999-06-01")["Close"]
kospi = pd.Series(_k.to_numpy(dtype=float), index=pd.to_datetime(_k.index))

FRED = ("https://fred.stlouisfed.org/graph/fredgraph.csv"
        "?id=IR3TIB01KRM156N&cosd=1999-01-01")
try:                                    # 한국 3개월 은행간금리 — 현금 보유 시 수익률
    _r = pd.read_csv(FRED, parse_dates=["observation_date"]).set_index("observation_date")
    cash_rate = (_r.iloc[:, 0].astype(float) / 100.0).dropna()
    print(f"KOSPI {kospi.index[-1].date()} {kospi.iloc[-1]:,.0f} · "
          f"현금금리 최근 {cash_rate.iloc[-1]:.2%}")
except Exception as exc:
    cash_rate = 0.0
    print("현금금리 로딩 실패 → 0% 가정:", exc)

## 4. 전략 엔진

모든 점수는 **판정 시점 이전** 데이터만 사용합니다(후행편향 없음).
추세필터도 전일 종가까지의 정보로 판정합니다.

In [ ]:
#@title 점수 · 추세필터 · 백테스트 엔진
TRADING_DAYS = 250

def score_matrix(kind, px, am, look):
    '''리밸런싱 시점까지의 정보만으로 계산한 주도주 점수.'''
    if kind == "turnover":               # 최근 거래대금 = 시장의 관심
        return am.rolling(look, min_periods=look // 2).mean()
    if kind == "turnover_growth":        # 관심이 '늘고 있는' 종목
        return (am.rolling(look, min_periods=look // 2).mean()
                / am.rolling(look * 4, min_periods=look * 2).mean())
    if kind == "mom":                    # 12-1개월 모멘텀 (최근 1개월 제외)
        return px.shift(20) / px.shift(look) - 1.0
    if kind == "mom_raw":                # 최근 1개월 포함 모멘텀
        return px / px.shift(look) - 1.0
    if kind == "combo":                  # 거래대금 순위 + 모멘텀 순위
        return (am.rolling(look, min_periods=look // 2).mean().rank(axis=1, pct=True)
                + (px.shift(20) / px.shift(look) - 1.0).rank(axis=1, pct=True))
    raise ValueError(kind)


def regime_filter(kospi, idx, ma):
    '''KOSPI가 ma일 이동평균 위일 때만 주식 보유. 전일 종가까지만 사용.'''
    if kospi is None or ma <= 0:
        return np.ones(len(idx), dtype=bool)
    ok = (kospi > kospi.rolling(ma, min_periods=ma).mean()).shift(1)
    return (ok.reindex(idx.union(ok.index)).ffill()
              .reindex(idx).fillna(True).to_numpy(bool))


def daily_cash_factor(rate, idx):
    if isinstance(rate, float):
        ann = np.full(len(idx), rate)
    else:
        ann = (rate.reindex(idx.union(rate.index)).ffill()
                   .reindex(idx).ffill().bfill().to_numpy())
    return 1.0 + ann / TRADING_DAYS


def run_rotation(px, mc, am, kind, top_k, uni, months, look, cost_bps,
                 start, end, kospi=None, ma=0, rate=0.0, stop=0.0, cache=None):
    '''모멘텀/거래대금 상위 K종목 동일비중 로테이션.'''
    idx = px.index[(px.index >= start) & (px.index <= end)]
    if len(idx) < 60:
        return None, None

    cache = {} if cache is None else cache
    key = (kind, look, idx[0], idx[-1])
    if key not in cache:
        cache[key] = score_matrix(kind, px, am, look).reindex(idx).to_numpy(np.float32)
    SC = cache[key]
    bkey = ("base", idx[0], idx[-1])
    if bkey not in cache:
        R = np.nan_to_num(px.pct_change().reindex(idx).to_numpy(np.float32), nan=0.0,
                          posinf=0.0, neginf=0.0)
        cache[bkey] = (R,
                       mc.rank(axis=1, ascending=False).reindex(idx).to_numpy(np.float32),
                       px.notna().cumsum().reindex(idx).to_numpy(np.int32),
                       px.reindex(idx).notna().to_numpy())
    R, MR, AGE, OKPX = cache[bkey]

    month_first = pd.Series(idx).groupby(idx.to_period("M")).first().sort_values()
    rebal = set(pd.DatetimeIndex(month_first.to_numpy())[::months])
    is_rebal = np.array([d in rebal for d in idx])
    ok_regime = regime_filter(kospi, idx, ma)
    cf_ = daily_cash_factor(rate, idx)
    cost = cost_bps / 10000.0

    n = px.shape[1]
    w = np.zeros(n)
    mult, peak = np.ones(n), np.zeros(n)
    equity = np.empty(len(idx))
    invested = np.zeros(len(idx), dtype=bool)
    val, last_pick, picks = 1.0, np.array([], dtype=int), []

    for i in range(len(idx)):
        if i > 0:
            if w.sum() > 0:
                gross = w * (1.0 + R[i])
                tot = gross.sum()
                if tot > 0:
                    val *= tot
                    w = gross / tot
                held = w > 0
                mult[held] *= (1.0 + R[i][held])
                peak[held] = np.maximum(peak[held], mult[held])
            else:
                val *= cf_[i]

        if is_rebal[i]:
            elig = (MR[i] <= uni) & (AGE[i] >= MIN_HISTORY) & OKPX[i] & np.isfinite(SC[i])
            cand = np.flatnonzero(elig)
            if len(cand) >= max(3, top_k // 2):
                last_pick = cand[np.argsort(-SC[i][cand])][:top_k]
                picks.append((idx[i], [px.columns[j] for j in last_pick]))

        if stop > 0 and w.sum() > 0:                      # 개별 트레일링 스탑
            hit = (w > 0) & (mult < peak * (1 - stop))
            if hit.any():
                keep = w.copy()
                keep[hit] = 0.0
                val *= (1.0 - float(w[hit].sum()) * cost)
                w = keep / keep.sum() if keep.sum() > 0 else keep

        want = bool(ok_regime[i])
        new = None
        if want and len(last_pick) and (is_rebal[i] or w.sum() == 0):
            new = np.zeros(n)
            new[last_pick] = 1.0 / len(last_pick)
        elif not want and w.sum() > 0:
            new = np.zeros(n)
        if new is not None:
            val *= (1.0 - float(np.abs(new - w).sum()) * cost)
            entering = (new > 0) & (w == 0)
            mult[entering], peak[entering] = 1.0, 1.0
            w = new

        invested[i] = w.sum() > 0
        equity[i] = val

    eq = pd.Series(equity, index=idx)
    eq.attrs["exposure"] = float(invested.mean())
    eq.attrs["in_market"] = pd.Series(invested, index=idx)
    return eq, picks


def cagr(s):
    yrs = (s.index[-1] - s.index[0]).days / 365.25
    return (s.iloc[-1] / s.iloc[0]) ** (1 / yrs) - 1 if yrs > 0 else np.nan

def mdd(s):
    return float((s / s.cummax() - 1).min())

def sharpe(s):
    r = s.pct_change().dropna()
    return float(r.mean() / r.std() * np.sqrt(TRADING_DAYS)) if r.std() else np.nan

print("엔진 준비 완료")

## 5. 파라미터

슬라이더를 움직이고 이 셀 아래를 다시 실행하면 결과가 바뀝니다.
기본값은 2000~2026 검증에서 채택한 조합입니다.

In [ ]:
#@title 전략 설정 { run: "auto", display-mode: "form" }
KIND = "mom" #@param ["mom", "mom_raw", "combo", "turnover", "turnover_growth"]
TOP_K = 10 #@param {type:"slider", min:3, max:30, step:1}
UNIVERSE = 100 #@param {type:"slider", min:50, max:300, step:10}
REBAL_MONTHS = 1 #@param [1, 2, 3, 6, 12] {type:"raw"}
LOOKBACK = 250 #@param {type:"slider", min:60, max:500, step:10}
MA_FILTER = 200 #@param {type:"slider", min:0, max:300, step:10}
COST_BPS = 25 #@param {type:"slider", min:0, max:150, step:5}
STOP_LOSS = 0 #@param {type:"slider", min:0, max:50, step:5}
START = "2000-01-01" #@param {type:"string"}

CFG = dict(kind=KIND, top_k=TOP_K, uni=UNIVERSE, months=int(REBAL_MONTHS),
           look=LOOKBACK, cost_bps=COST_BPS, ma=MA_FILTER, stop=STOP_LOSS / 100)
END = str(px.index[-1].date())
print(f"모멘텀 룩백 {LOOKBACK}일 · 시총 {UNIVERSE}위 이내 · 상위 {TOP_K}종목 · "
      f"{REBAL_MONTHS}개월 리밸런싱 · MA{MA_FILTER} 필터 · 편도 {COST_BPS}bp")

## 6. 백테스트 실행

In [ ]:
#@title 실행 및 요약
cache = {}
eq, picks = run_rotation(px, mc, am, CFG["kind"], CFG["top_k"], CFG["uni"],
                         CFG["months"], CFG["look"], CFG["cost_bps"], START, END,
                         kospi=kospi, ma=CFG["ma"], rate=cash_rate,
                         stop=CFG["stop"], cache=cache)

eq_nofilter, _ = run_rotation(px, mc, am, CFG["kind"], CFG["top_k"], CFG["uni"],
                              CFG["months"], CFG["look"], CFG["cost_bps"], START, END,
                              ma=0, rate=cash_rate, stop=CFG["stop"], cache=cache)

ks = kospi[(kospi.index >= eq.index[0]) & (kospi.index <= eq.index[-1])]
mrank = mc.rank(axis=1, ascending=False)
big = mrank.loc[eq.index[0]].nsmallest(CFG["top_k"]).index.tolist()
_b = px.loc[eq.index, big]
bh = _b.div(_b.bfill().iloc[0]).ffill().mean(axis=1)

curves = {"로테이션 전략": eq, "KOSPI": ks, f"시총 TOP{CFG['top_k']} 매수후보유": bh}

print(f"{'전략':<28}{'누적':>12}{'CAGR':>9}{'MDD':>9}{'샤프':>8}")
print("─" * 68)
for lbl, s in curves.items():
    print(f"{lbl:<28}{(s.iloc[-1]/s.iloc[0]-1)*100:>11,.0f}%{cagr(s)*100:>8.1f}%"
          f"{mdd(s)*100:>8.0f}%{sharpe(s):>8.2f}")
print(f"{'  └ 추세필터 제거 시':<28}{(eq_nofilter.iloc[-1]-1)*100:>11,.0f}%"
      f"{cagr(eq_nofilter)*100:>8.1f}%{mdd(eq_nofilter)*100:>8.0f}%{sharpe(eq_nofilter):>8.2f}")
print(f"\n주식 보유일 비율 {eq.attrs['exposure']:.0%} · 리밸런싱 {len(picks)}회", end="")
if len(picks) > 1:
    t = np.mean([len(set(picks[i][1]) ^ set(picks[i-1][1])) / 2 / CFG["top_k"]
                 for i in range(1, len(picks))])
    print(f" · 회당 교체율 {t:.0%} · 연 회전율 약 {t * 12 / CFG['months'] * 100:.0f}%")

In [ ]:
#@title 기간별 성과
PERIODS = [("2000-01-01", "2004-12-31"), ("2005-01-01", "2009-12-31"),
           ("2010-01-01", "2014-12-31"), ("2015-01-01", "2019-12-31"),
           ("2020-01-01", "2024-12-31"), ("2025-01-01", "2026-12-31")]

print(f"{'기간':<12}{'전략':>11}{'KOSPI':>11}{'전략MDD':>11}{'KOSPI MDD':>11}")
print("─" * 58)
for a, b in PERIODS:
    w = eq[(eq.index >= a) & (eq.index <= b)]
    if len(w) < 60:
        continue
    k = ks[(ks.index >= w.index[0]) & (ks.index <= w.index[-1])]
    print(f"{a[:4]}~{b[:4]}   {(w.iloc[-1]/w.iloc[0]-1)*100:>10.0f}%"
          f"{(k.iloc[-1]/k.iloc[0]-1)*100:>10.0f}%{mdd(w)*100:>10.0f}%{mdd(k)*100:>10.0f}%")

## 7. 차트

In [ ]:
#@title 차트 공통 설정
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e7e6e2"
BLUE, ORANGE, AQUA, SHADE = "#2a78d6", "#eb6834", "#1baf7a", "#eceae5"

def style_ax(ax, title="", ylabel=""):
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, lw=1, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK2, length=0, labelsize=10)
    if title:
        ax.set_title(title, color=INK, fontsize=13, pad=14, loc="left")
    if ylabel:
        ax.set_ylabel(ylabel, color=INK2, fontsize=10)
    return ax

def new_axes(h=5.2, title="", ylabel=""):
    fig, ax = plt.subplots(figsize=(11, h), facecolor=SURFACE)
    style_ax(ax, title, ylabel)
    return fig, ax

def end_labels(ax, items, gap=0.065):
    # 선 끝 직접 라벨. 겹치면 세로로 밀어낸다. 텍스트는 잉크 색 - 색은 선이 담당한다.
    lo, hi = ax.get_ylim()
    log = ax.get_yscale() == "log"
    def frac(v):
        if log:
            return (np.log10(v) - np.log10(lo)) / (np.log10(hi) - np.log10(lo))
        return (v - lo) / (hi - lo)
    rows = sorted(((frac(float(s.iloc[-1])), t) for s, t in items), key=lambda r: r[0])
    ys = [r[0] for r in rows]
    for i in range(1, len(ys)):
        ys[i] = max(ys[i], ys[i - 1] + gap)
    shift = max(0.0, max(ys) - 0.97)
    for (_, t), y in zip(rows, ys):
        ax.annotate(t, xy=(1.015, y - shift), xycoords="axes fraction", va="center",
                    fontsize=10, color=INK, annotation_clip=False)

def finish(fig, note="", right=0.79, bottom=0.15):
    # 오른쪽에 직접 라벨 자리를 비워두고, 필요하면 하단 주석을 단다.
    fig.subplots_adjust(left=0.07, right=right, top=0.89, bottom=bottom, hspace=0.18)
    if note:
        fig.text(0.07, 0.035, note, fontsize=9, color=INK2)
    plt.show()

def spans(flags, index, min_len=15):
    # 연속으로 False인 구간 중 min_len 이상인 것만 (시작, 끝)으로 반환
    out, start = [], None
    for i, on in enumerate(flags):
        if not on and start is None:
            start = i
        elif on and start is not None:
            if i - start >= min_len:
                out.append((index[start], index[i]))
            start = None
    if start is not None and len(flags) - start >= min_len:
        out.append((index[start], index[-1]))
    return out

In [ ]:
#@title 자산곡선 (로그 스케일)
fig, ax = new_axes(5.6, "자산곡선 — 1을 투자했을 때의 배수 (로그 스케일)")

# 추세필터로 현금이었던 구간을 옅게 깔아 '가만히 있기'를 드러낸다
for a_, b_ in spans(eq.attrs["in_market"].to_numpy(), eq.index):
    ax.axvspan(a_, b_, color=SHADE, lw=0, zorder=0)

items, top = [], 0
for s, color, lbl in ((eq, BLUE, "로테이션 전략"),
                      (bh / bh.iloc[0], AQUA, f"시총 TOP{CFG['top_k']} 보유"),
                      (ks / ks.iloc[0], ORANGE, "KOSPI")):
    ax.plot(s.index, s.to_numpy(), color=color, lw=2, zorder=3,
            solid_capstyle="round", label=lbl)
    items.append((s, f"{lbl}  {s.iloc[-1]:,.0f}배"))
    top = max(top, float(s.max()))

ax.set_yscale("log")
ticks = [t for t in (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000) if t <= top * 1.5]
ax.set_yticks(ticks)
ax.set_yticklabels([f"{t:,}배" for t in ticks])
ax.set_xlim(eq.index[0], eq.index[-1])
ax.legend(frameon=False, loc="upper left", fontsize=10, labelcolor=INK2)
end_labels(ax, items)
finish(fig, "음영 = 추세필터가 꺼져 전량 현금이던 구간 (15거래일 이상)")

In [ ]:
#@title 낙폭 (Drawdown) — 위아래로 나눠 비교
fig, axs = plt.subplots(2, 1, figsize=(11, 5.8), sharex=True, sharey=True,
                        facecolor=SURFACE)
floor = min(mdd(eq), mdd(ks)) * 108

for ax, (s, color, lbl) in zip(axs, ((eq, BLUE, "로테이션 전략"), (ks, ORANGE, "KOSPI"))):
    style_ax(ax)
    dd = (s / s.cummax() - 1) * 100
    ax.fill_between(dd.index, dd.to_numpy(), 0, color=color, alpha=0.18, lw=0, zorder=2)
    ax.plot(dd.index, dd.to_numpy(), color=color, lw=1.6, zorder=3)
    ax.set_ylim(floor, 4)
    ax.yaxis.set_major_formatter(lambda v, p: f"{v:.0f}%")
    ax.annotate(f"{lbl}   최악 {dd.min():.0f}%", xy=(0.008, 0.12),
                xycoords="axes fraction", fontsize=11, color=INK, fontweight="medium",
                bbox=dict(facecolor=SURFACE, edgecolor="none", pad=3))

axs[0].set_title("고점 대비 낙폭", color=INK, fontsize=13, pad=14, loc="left")
axs[0].set_xlim(eq.index[0], eq.index[-1])
finish(fig, right=0.97)

In [ ]:
#@title 연도별 수익률
ye = eq.resample("YE").last().pct_change().dropna() * 100
yk = ks.resample("YE").last().pct_change().dropna() * 100
yrs = sorted(set(ye.index.year) & set(yk.index.year))
a = np.array([ye[ye.index.year == y].iloc[0] for y in yrs])
b = np.array([yk[yk.index.year == y].iloc[0] for y in yrs])

fig, ax = new_axes(4.8, "")
x = np.arange(len(yrs)); w = 0.40
ax.bar(x - w / 2, a, w * 0.94, color=BLUE, zorder=3, label="로테이션 전략")
ax.bar(x + w / 2, b, w * 0.94, color=ORANGE, zorder=3, label="KOSPI")
ax.axhline(0, color=INK2, lw=1, zorder=4)
ax.set_xticks(x); ax.set_xticklabels([str(y)[2:] for y in yrs], fontsize=9)
ax.set_xlim(-0.8, len(yrs) - 0.2)
ax.yaxis.set_major_formatter(lambda v, p: f"{v:+.0f}%")
ax.set_title("연도별 수익률", color=INK, fontsize=13, pad=34, loc="left")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.005), ncol=2,
          fontsize=10, labelcolor=INK2)
finish(fig, f"전략 승 {int((a > b).sum())}년 / {len(yrs)}년", right=0.97)

In [ ]:
#@title 롤링 3년 CAGR — 언제 통했고 언제 안 통했나
W = 750
r_eq = eq.rolling(W).apply(lambda v: (v[-1] / v[0]) ** (250 / W) - 1, raw=True).dropna() * 100
r_ks = ks.reindex(eq.index).ffill().rolling(W).apply(
    lambda v: (v[-1] / v[0]) ** (250 / W) - 1, raw=True).dropna() * 100
i = r_eq.index.intersection(r_ks.index)

fig, ax = new_axes(4.6, "3년 롤링 CAGR")
ax.fill_between(i, r_eq[i].to_numpy(), r_ks[i].to_numpy(),
                where=(r_eq[i] >= r_ks[i]).to_numpy(), color=BLUE, alpha=0.13, lw=0, zorder=2)
ax.fill_between(i, r_eq[i].to_numpy(), r_ks[i].to_numpy(),
                where=(r_eq[i] < r_ks[i]).to_numpy(), color=ORANGE, alpha=0.13, lw=0, zorder=2)
items = []
for s, color, lbl in ((r_eq[i], BLUE, "로테이션 전략"), (r_ks[i], ORANGE, "KOSPI")):
    ax.plot(s.index, s.to_numpy(), color=color, lw=2, zorder=3, label=lbl)
    items.append((s, lbl))
ax.axhline(0, color=INK2, lw=1, zorder=4)
ax.set_xlim(i[0], i[-1])
ax.yaxis.set_major_formatter(lambda v, p: f"{v:+.0f}%")
ax.legend(frameon=False, loc="upper left", fontsize=10, labelcolor=INK2)
end_labels(ax, items)
finish(fig, f"전략 우위 구간 비율 {(r_eq[i] > r_ks[i]).mean():.0%}")

## 8. 현재 시점 — 지금 이 전략이 시키는 것

In [ ]:
#@title 추세 상태와 편입 후보
day = px.index[-1]
print(f"기준일 {day.date()}\n")

if CFG["ma"]:
    k_now = float(kospi.iloc[-1]); k_ma = float(kospi.rolling(CFG["ma"]).mean().iloc[-1])
    on = k_now > k_ma
    print(f"추세필터  KOSPI {k_now:,.0f}  vs  MA{CFG['ma']} {k_ma:,.0f}  ({k_now/k_ma-1:+.1%})")
    print(f"          → {'주식 보유' if on else '전량 현금 (편입 후보는 참고용)'}\n")

sc = score_matrix(CFG["kind"], px, am, CFG["look"])
elig = ((mrank.loc[day] <= CFG["uni"]) & (px.notna().cumsum().loc[day] >= MIN_HISTORY)
        & px.loc[day].notna() & sc.loc[day].notna())
cand = sc.loc[day][elig].sort_values(ascending=False).head(CFG["top_k"])
mom12 = px.loc[day] / px.shift(CFG["look"]).loc[day] - 1.0

print(f"편입 후보 — 동일비중 {100/CFG['top_k']:.1f}%씩")
print(f"{'#':<4}{'종목':<18}{'시총순위':>9}{'12개월 수익률':>15}")
print("─" * 48)
for n_, (code_, _) in enumerate(cand.items(), 1):
    print(f"{n_:<4}{names.get(code_, code_)[:16]:<18}"
          f"{int(mrank.loc[day, code_]):>9}{mom12.get(code_, np.nan)*100:>14.1f}%")

## 9. 파라미터 민감도

**단일 최적 조합을 찾는 셀이 아닙니다.** 각 손잡이를 흔들었을 때
성과가 완만한 언덕인지 뾰족한 스파이크인지 보는 셀입니다.
스파이크면 과최적화를 의심해야 합니다. 1~2분 걸립니다.

In [ ]:
#@title 한 축씩 흔들기
axes_to_scan = {
    "top_k":  [5, 7, 10, 15, 20],
    "months": [1, 2, 3, 6],
    "ma":     [0, 120, 150, 200, 250],
    "look":   [120, 200, 250, 300, 400],
    "uni":    [60, 80, 100, 120, 150, 200],
}
base_eq = eq
print(f"기준  CAGR {cagr(base_eq)*100:.1f}%  MDD {mdd(base_eq)*100:.0f}%  "
      f"샤프 {sharpe(base_eq):.2f}\n")
print(f"{'축':<8}" + "".join(f"{'값':>6}{'CAGR':>8}{'샤프':>7}" for _ in range(1))[:0])
for axis, values in axes_to_scan.items():
    cells = []
    for v in values:
        c = dict(CFG); c[axis] = v
        e, _ = run_rotation(px, mc, am, c["kind"], c["top_k"], c["uni"], c["months"],
                            c["look"], c["cost_bps"], START, END, kospi=kospi,
                            ma=c["ma"], rate=cash_rate, stop=c["stop"], cache=cache)
        cells.append(f"{v}:{cagr(e)*100:.0f}%/{sharpe(e):.2f}")
    print(f"  {axis:<8}" + "   ".join(cells))
print("\n표기 = 파라미터값 : CAGR / 샤프")

In [ ]:
#@title 거래비용 스트레스
print(f"{'편도비용':<10}{'CAGR':>9}{'MDD':>9}{'샤프':>8}")
print("─" * 38)
for c_ in (0, 25, 50, 75, 100, 150):
    e, _ = run_rotation(px, mc, am, CFG["kind"], CFG["top_k"], CFG["uni"], CFG["months"],
                        CFG["look"], c_, START, END, kospi=kospi, ma=CFG["ma"],
                        rate=cash_rate, stop=CFG["stop"], cache=cache)
    print(f"{c_:>5}bp   {cagr(e)*100:>8.1f}%{mdd(e)*100:>8.0f}%{sharpe(e):>8.2f}")
print("\n한국 대형주 실제 비용: 매수 1.5bp / 매도 16.5bp(수수료+거래세 0.15%)")
print("슬리피지 포함 편도 12~22bp — 기본값 25bp는 보수적인 가정")

In [ ]:
#@title 변동성 매칭 — MDD를 낮추려면 얼마나 태워야 하나
r_s = eq.pct_change().fillna(0).to_numpy()
r_c = daily_cash_factor(cash_rate, eq.index) - 1.0

print(f"{'전략 비중':<10}{'현금':>8}{'CAGR':>9}{'MDD':>9}{'샤프':>8}")
print("─" * 46)
for x in (1.0, 0.85, 0.70, 0.55, 0.40):
    s = pd.Series(np.cumprod(1 + x * r_s + (1 - x) * r_c), index=eq.index)
    print(f"{x:>7.0%}   {1-x:>7.0%}{cagr(s)*100:>8.1f}%{mdd(s)*100:>8.0f}%{sharpe(s):>8.2f}")
print(f"\n참고: KOSPI 매수후보유  CAGR {cagr(ks)*100:.1f}%  MDD {mdd(ks)*100:.0f}%  "
      f"샤프 {sharpe(ks):.2f}")
print("샤프가 높은 전략은 비중을 줄여 MDD를 맞추는 쪽이 유리하다.")

## 부록. 원래 영상 전략과의 비교

**"고점 대비 −20% 하락 후, 저점에서 하락폭의 절반을 회복하면 매수"** — 이 진입 규칙에
정말 정보가 있는지 확인합니다.

표본은 **각 기간 시작일의 시총 상위 20종목**입니다. 지금 잘나가는 종목을 고르면
"이미 오른 종목에서 눌림목 매수가 통했다"는 당연한 결과가 나오므로,
**그 시점에 알 수 있었던 정보만으로** 표본을 정합니다.

검정 두 가지:
1. **이벤트 스터디** — 시그널 날 산 것 vs 같은 종목을 아무 날에나 산 것
2. **몬테카를로** — 같은 종목·같은 거래횟수·같은 보유일수를 무작위 시점에 배치했을 때의 분포

In [ ]:
#@title 부록 A. 편향 없는 표본 만들기
PERIOD_STARTS = ["2000-01-01", "2005-01-01", "2010-01-01",
                 "2015-01-01", "2020-01-01", "2025-01-01"]

test_codes = []
for ds in PERIOD_STARTS:
    days = px.index[px.index >= ds]
    if not len(days):
        continue
    top20 = mrank.loc[days[0]].dropna().nsmallest(20).index.tolist()
    test_codes += [c for c in top20 if c in px.columns]
test_codes = list(dict.fromkeys(test_codes))
print(f"표본 {len(test_codes)}종목 — 6개 기간 시작일의 시총 상위 20종목 합집합")
print(", ".join(names.get(c, c) for c in test_codes[:24]), "…")

In [ ]:
#@title 부록 B. 이벤트 스터디 — 시그널에 정보가 있는가
DROP, REC, HOLD = 0.20, 0.50, 250   # 하락 트리거 / 회복 비율 / 보유 거래일

def dip_trades(p, drop=DROP, rec=REC, hold=HOLD):
    # 실시간 상태기계. 보유 중에는 새 시그널을 받지 않으므로 구간이 겹치지 않는다.
    out, n, i = [], len(p), 0
    state, peak, trough = "WATCH", p[0], np.inf
    while i < n:
        v = p[i]
        if state == "WATCH":
            peak = max(peak, v)
            if v <= peak * (1 - drop):
                state, trough = "ARMED", v
        else:
            trough = min(trough, v)
            if v >= trough + rec * (peak - trough):
                ln = min(hold, n - 1 - i)
                if ln <= 0:
                    break
                out.append((i, ln))
                i += ln
                state, peak, trough = "WATCH", p[min(i, n - 1)], np.inf
                continue
        i += 1
    return out

HORIZONS = (20, 60, 120, 250)
hit = {h: [] for h in HORIZONS}
base = {h: [] for h in HORIZONS}
series = {}
for code_ in test_codes:
    s = px[code_].dropna()
    s = s[s.index >= START]
    if len(s) < 750:
        continue
    p = s.to_numpy(float)
    series[code_] = p
    for i, _ in dip_trades(p):
        for h in HORIZONS:
            if i + h < len(p):
                hit[h].append(p[i + h] / p[i] - 1)
    for h in HORIZONS:
        base[h].extend((p[h:] / p[:-h] - 1).tolist())

print(f"검증 {len(series)}종목 · 시그널 {len(hit[20])}건\n")
print(f"{'보유':<10}{'시그널 평균':>12}{'기준 평균':>11}{'시그널 중앙':>12}"
      f"{'기준 중앙':>11}{'시그널 승률':>12}{'기준 승률':>11}")
print("-" * 82)
for h in HORIZONS:
    v, b_ = np.array(hit[h]), np.array(base[h])
    if not len(v):
        continue
    print(f"{h}거래일{'':<4}{v.mean()*100:>11.1f}%{b_.mean()*100:>10.1f}%"
          f"{np.median(v)*100:>11.1f}%{np.median(b_)*100:>10.1f}%"
          f"{(v>0).mean()*100:>11.1f}%{(b_>0).mean()*100:>10.1f}%")
print("\n'기준'은 같은 종목의 모든 날에서 계산한 h일 수익률 — 종목 효과는 이미 통제됨.")
print("승률·중앙값이 기준을 넘지 못하면 그 진입 규칙에 정보가 없다는 뜻.")

In [ ]:
#@title 부록 C. 몬테카를로 — 무작위 타이밍과 비교 (결정적 검정)
N_SIMS = 1000
rng = np.random.default_rng(20260920)

def windows_value(p, wins, cost=0.0025):
    v = 1.0
    for st, ln in wins:
        en = min(st + ln, len(p) - 1)
        if en > st:
            v *= (p[en] / p[st]) * (1 - cost) ** 2
    return v

def random_windows(n, runs, rng):
    # 실제와 같은 개수·같은 길이의 보유구간을 무작위 시점에 비중첩 배치
    slack = max(n - 1 - sum(runs), 0)
    cuts = np.sort(rng.integers(0, slack + 1, size=len(runs)))
    gaps = cuts - np.concatenate(([0], cuts[:-1]))
    wins, prev = [], 0
    for gap, ln in zip(gaps, runs):
        st = prev + int(gap)
        wins.append((st, ln))
        prev = st + ln
    return wins

rows = []
for code_, p in series.items():
    tr = dip_trades(p)
    if not tr:
        continue
    runs = [ln for _, ln in tr]
    act = windows_value(p, tr)
    sims = np.array([windows_value(p, random_windows(len(p), runs, rng))
                     for _ in range(N_SIMS)])
    rows.append((code_, act, sims, float((sims < act).mean() * 100)))

pcts = np.array([r[3] for r in rows])
print(f"검증 {len(rows)}종목 · 종목당 시뮬레이션 {N_SIMS}회\n")
print("종목별로 '실제 시그널 타이밍'이 그 종목의 무작위 타이밍 분포에서 몇 번째인지:\n")
print(f"  백분위 중앙값      {np.median(pcts):.0f}")
print(f"  백분위 평균        {pcts.mean():.0f}")
print(f"  무작위보다 나은 종목 {(pcts > 50).sum()}/{len(pcts)}  ({(pcts > 50).mean():.0%})")
print(f"  분포  하위25% {np.percentile(pcts, 25):.0f} · 상위25% {np.percentile(pcts, 75):.0f}")

# 동일비중 포트폴리오 관점 (기하평균 — 한 종목이 결과를 지배하지 않도록)
geo_act = float(np.exp(np.mean(np.log([r[1] for r in rows]))))
geo_sim = np.exp(np.mean(np.log(np.vstack([r[2] for r in rows])), axis=0))
print(f"\n종목 기하평균 최종배수  실제 {geo_act:.2f}배  vs  "
      f"무작위 중앙값 {np.median(geo_sim):.2f}배 "
      f"(5% {np.percentile(geo_sim, 5):.2f} / 95% {np.percentile(geo_sim, 95):.2f})")
print(f"→ 백분위 {float((geo_sim < geo_act).mean() * 100):.0f}")
print("\n백분위 50 근처 = 진입 타이밍 자체에 정보가 없다는 뜻.")
print("같은 종목·같은 거래횟수·같은 보유일수라 종목 효과는 완전히 통제된다.")

---

## 정리

이 전략이 KOSPI를 이긴 이유는 **타이밍이 아니라 종목선정**입니다. 그리고 그 우위는
**시기를 심하게 탑니다** — 2000~2013에 집중되어 있고, 2013년 7월 이후로는
KOSPI 대비 샤프 우위가 사라집니다.

### 실행한다면

- 전액 말고 **주식자산의 ~70%**. 위 변동성 매칭 셀 참조
- 연 회전율 380% → **ISA·연금계좌** 권장
- 편입 종목이 한 섹터(현재는 반도체·전력기기)로 쏠릴 수 있음 — 분산이 안 된다는 뜻

### 한계

배당 미반영(전략·벤치마크 모두, 전략에 불리) · 슬리피지와 유동성 제약 미반영 ·
시총 300위 진입 경험 종목으로 유니버스 제한 · 인적분할 종목의 분할 신설법인 가치 미반영 ·
26년이라는 표본은 독립적인 시장 국면으로 보면 5~6개에 불과함

### 데이터

[FinanceData/marcap](https://github.com/FinanceData/marcap) (KRX 전종목, 상장폐지 포함) ·
[FRED IR3TIB01KRM156N](https://fred.stlouisfed.org/series/IR3TIB01KRM156N) (한국 3M 은행간금리) ·
FinanceDataReader (KOSPI)